Brandon Jackson, Allyanna Panganiban, Karen Sabile
SYSEN 5211 Final Project
Spring 2026

In [18]:
# library imports
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import math
import random
import matplotlib.pyplot as plt

In [ ]:
shifts = ['Day', 'Evening', 'Night']
n_shifts = len(shifts)
num_nurses = 22
patients = [56, 30, 50]

m = gp.Model('hospital_lp')

# continuous decision variables: the number of nurses assigned to each shift
nurse_count = {}
for s, shift in enumerate(shifts):
    nurse_count[s] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name='{shift}')

# binary decision variables: three for every nurse (to cover all three shifts)
nurses = []
for n in range(num_nurses):
    nurses.append([])
    for shift in shifts:
        nurses[n].append(m.addVar(lb=0.0, vtype=GRB.BINARY, name='{n}_{shift}'))

# objective function: maximize minimum shift coverage
z = m.addVar(lb=0.0, name="min_ratio")

for s in range(n_shifts):
    m.addConstr(
        (gp.quicksum(nurses[n][s] for n in range(num_nurses)) / patients[s]) >= z
    )

m.setObjective(z, GRB.MAXIMIZE)

# each nurse assigned to one shift
for n in range(num_nurses):
    m.addConstr(gp.quicksum(nurses[n][s] for s in range(n_shifts)) <= 1)

# link continuous decision variables to binary decision variables
for s in range(n_shifts):
    m.addConstr(nurse_count[s] == gp.quicksum(nurses[n][s] for n in range(num_nurses)))

m.optimize()

# -----------------------------
# Results
# -----------------------------
if m.status == GRB.OPTIMAL:
    print("\n✅ Optimal solution found:")
    for s, shift in enumerate(shifts):
        print(f"  {shift} Shift: {math.ceil(nurse_count[s].X)} nurses")  # integer format
        print(f"     Nurse-to-Patient Ratio: 1:{1/(nurse_count[s].X/patients[s]):,.2f}")
    print(f"  Lowest Ratio: 1:{1/m.ObjVal:,.2f}")
else:
    print(f"❌ Optimization ended with status {m.status}.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.3.0 23D2057)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 28 rows, 70 columns and 204 nonzeros (Max)
Model fingerprint: 0xf3890042
Model has 1 linear objective coefficients
Variable types: 4 continuous, 66 integer (66 binary)
Coefficient statistics:
  Matrix range     [2e-02, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 25 rows, 67 columns, 135 nonzeros
Variable types: 1 continuous, 66 integer (66 binary)

Root relaxation: objective 1.617647e-01, 43 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    0.161